## Hack 指令格式：两种指令、两个寄存器

> 2026-10-22 · 来源：nand2tetris 项目复习（modules/408/NAND2TETRIS.md）
- A-指令 = `0` + 15 位常量（装入 A）；C-指令 = `111a cccccc ddd jjj`（`a` 位在 A/M 之间选运算源，comp 6 位，dest 3 位 = A/D/M，jump 3 位 = <0 / =0 / >0）
- 只有 **D（数据）** 与 **A（地址/数据）** 两个寄存器，`M` 恒指 `RAM[A]`
- **一条 C-指令里 A 与 M 不能同时当运算源**（被 `a` 位二选一限制）：想用 `M` 参与运算必须先 `D=M`
- **跳转目标必须先装进 A**：硬件上 PC 的输入只有 A 寄存器这一条路
- **没有标志寄存器**：跳转直接看 ALU 输出的 `zr`/`ng` 组合，不做「先算再判断别的事」
- 汇编器两遍扫描的根因是前向引用：第一遍只建标签表（`(LABEL)` 不推进 ROM 地址），第二遍才翻译并把变量从 RAM 16 起分配


## 栈帧是软件契约：call / return 的五个槽

> 2026-10-22 · 来源：nand2tetris 项目复习（modules/408/NAND2TETRIS.md）
- `call f k` 压入 5 槽：返回地址、LCL、ARG、THIS、THAT → 再令 `ARG = SP-5-k`、`LCL = SP`
- `return` 五步：`FRAME = LCL` → `RET = *(FRAME-5)` → `*ARG = 返回值`、`SP = ARG+1` → 恢复 THAT/THIS/ARG/LCL → 跳 RET
- ★ **无参数调用时 `ARG[0]` 就是 `FRAME-5`**：`RET` 必须在写返回值之前取走，否则返回地址被返回值覆盖（本章第一号杀手）
- 标签规则：`goto/label` 要加 `函数名$` 前缀，`function` 定义标签**不加**前缀；翻译 `return` 时不要重置当前函数名（一个函数可能有两个 return）
- `R13/R14/R15` 是易失的临时寄存器，任何被调用函数都可能改掉它们
- 对比 408 / x86：caller-save 与 callee-save 是同一问题的另一种答案；**硬件只看到栈在长长短短地变化，「函数」这个概念完全由软件约定支撑**


## 栈帧与返回地址：缓冲区溢出为什么能劫持控制流

> 2026-10-22 · 来源：CSAPP 项目复习（`modules/408/CSAPP.md`） §7.3（实践仓 `learning-408/csapp/target1/`，attacklab 靶子，cookie `0x59b997fa`）
`getbuf` 的栈帧自高向低：**返回地址**（回到 test）→ 保存的 %rbp → `buf[...]`。`Gets` 无边界检查，写满 `BUFFER_SIZE` 后继续写就会覆盖返回地址；`ret` 把它弹进 `%rip` → **控制流被数据劫持**。

**三级防御与被绕方式**（选择题可能问「哪种防御挡哪类攻击」）：
- **栈随机化 ASLR**：猜不到注入代码地址 → 绕法是用栈上相对位置 / 改用 ROP
- **栈不可执行 NX**：注入的字节码直接段错误 → 绕法是 **ROP**（用程序里现成的 `ret` 结尾小片段当积木，用栈上一串地址串起来）
- **金丝雀 canary**：返回地址前放随机值校验 → 本 lab 靶子没开

**408 对应**：栈帧布局、`call/ret` 对栈的作用、函数调用的机器级表示（必考）；ROP 的构造不考。


## 三种机器级形态：跳转表 / 递归 / 链表（bomblab 反推）

> 2026-10-22 · 来源：CSAPP 项目复习（`modules/408/CSAPP.md`） §7.2（已有产出 `learning-408/csapp/bomb/solutions`，phase1–6 + secret）
408 不考逆向，但考「给一段 GCC x86-64 汇编问它在干什么」。三种形态要一眼认出：
- **跳转表**：`jmp *0x402470(,%rax,8)` + 一段连续 8 字节地址区 → `switch`；**表要按小端读**
- **递归**：函数体内 `call` 自身 + 栈上保存参数/局部变量
- **链表遍历**：`mov 0x8(%rbx), %rax`（偏移 8 = next 指针）、`mov (%rax), %eax`（偏移 0 = val）

**参数约定**：`%rdi %rsi %rdx %rcx %r8 %r9` 传前 6 个整型/指针参数，第 7 个起走栈，返回值在 `%rax`。

**三种寻址**：`(%rbx)` 取值；`0x8(%rbx)` 取结构体字段；`(%rax,%rbx,4)` 基址+变址×比例 = 数组下标（比例因子 = 元素大小）。
